In [236]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "plotting":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / "plotting" / "single_curves"

import csv
import math
from statistics import NormalDist

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

In [237]:
plt.rcParams["font.family"] = "Charter"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Charter"
plt.rcParams["mathtext.it"] = "Charter:italic"
plt.rcParams["mathtext.bf"] = "Charter:bold"

PLOTS_TO_MAKE = [
    "spread_final_selected",
    "lbf_final_selected",
    "rware_final_selected",
]

X_KEY = "global_step"
Y_KEY = "train_return_mean"
SMOOTHING_WINDOW = 250
CI_LEVEL = 0.95
DPI = 300

MERGE_IDENTICAL_ADJACENT_STAGES = True
PRINT_LEGEND = True
COMPACT_Y_LIMITS = True
LEGEND_FONTSIZE = 18
STAGE_FONTSIZE = 15
AXES_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 25

FIG_HEIGHT = 5.0
FIG_WIDTH = 9.0

Y_LABEL = "Mean training return"
X_LABEL = "Environment steps ($\\times 1e5$)"

SERIES_ORDER = ("ippo", "mappo", "pimac_v0", "pimac_v6")
ALGORITHM_LABELS = {
    "ippo": "IPPO",
    "mappo": "MAPPO",
    "pimac_v0": "PIC-MAPPO",
    "pimac_v6": "PC3D",
}
ALGORITHM_COLORS = {
    "ippo": "royalblue",
    "mappo": "#005d5d",
    "pimac_v0": "goldenrod",
    "pimac_v6": "#9f1853",
}

SELECTED_CONFIGS = {
    "lbf_final_selected": {
        "title": "LBF",
        "output_filename": "lbf_hard_final_learning_curves.png",
        "results_root": "results/lbf_hard",
        "run_prefix": "final_lbf_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "rware_final_selected": {
        "title": "RWARE",
        "output_filename": "rware_final_learning_curves.png",
        "results_root": "results/rware",
        "run_prefix": "final_robotic_warehouse_dynamic",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "spread_final_selected": {
        "title": "Spread",
        "output_filename": "spread_hard_final_learning_curves.png",
        "results_root": "results/spread",
        "run_prefix": "final_simple_spread_dynamic_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_03",
    },
}

STAGE_FACE_COLORS = ("#000000", "#4c78a8", "#f58518", "#54a24b", "#b279a2")
STAGE_ALPHA = 0.075
SEPARATOR_ALPHA = 0.42
LINE_Y_PADDING_FRACTION = 0.06


In [238]:
def rolling_mean(values, window):
    values = np.asarray(values, dtype=np.float64)
    if window <= 1:
        return values
    out = np.empty_like(values)
    cumsum = np.cumsum(values, dtype=np.float64)
    for i in range(len(values)):
        start = max(0, i - window + 1)
        total = cumsum[i] - (cumsum[start - 1] if start else 0.0)
        out[i] = total / (i - start + 1)
    return out


def load_history(path):
    rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
    x = np.asarray([float(row[X_KEY]) for row in rows], dtype=np.float64)
    y = np.asarray([float(row[Y_KEY]) for row in rows], dtype=np.float64)
    y = rolling_mean(y, SMOOTHING_WINDOW)
    return x, y


def parse_stage_descriptor(name):
    if "__" not in name:
        text = name.replace("stage_", "")
        return ((int(text),) if text.isdigit() else tuple(), tuple())
    stage_prefix, counts_suffix = name.split("__", maxsplit=1)
    stage_text = stage_prefix.replace("stage_", "")
    counts = tuple(int(x) for x in counts_suffix.split("_") if x)
    return ((int(stage_text),), counts)


def stage_label(stage_indices, counts):
    return rf"$n \in \{{{','.join(str(c) for c in counts)}\}}$"


def load_stage_segments(path):
    rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
    segments = []
    current_index = int(rows[0]["stage_index"])
    current_name = rows[0]["stage_name"]
    start_step = float(rows[0][X_KEY])
    previous_step = start_step

    for row in rows[1:]:
        stage_index = int(row["stage_index"])
        step = float(row[X_KEY])
        if stage_index != current_index:
            indices, counts = parse_stage_descriptor(current_name)
            indices = indices or (current_index,)
            segments.append({"indices": indices, "counts": counts, "start": start_step, "end": step, "label": stage_label(indices, counts)})
            current_index = stage_index
            current_name = row["stage_name"]
            start_step = step
        previous_step = step

    indices, counts = parse_stage_descriptor(current_name)
    indices = indices or (current_index,)
    segments.append({"indices": indices, "counts": counts, "start": start_step, "end": previous_step, "label": stage_label(indices, counts)})
    return segments


def merge_adjacent_stage_segments(segments):
    merged = []
    for segment in segments:
        if merged and segment["counts"] and merged[-1]["counts"] == segment["counts"] and segment["indices"][0] == merged[-1]["indices"][-1] + 1:
            merged[-1]["indices"] = merged[-1]["indices"] + segment["indices"]
            merged[-1]["end"] = segment["end"]
            merged[-1]["label"] = stage_label(merged[-1]["indices"], segment["counts"])
        else:
            merged.append(dict(segment))
    return merged

def aggregate_runs(paths):
    runs = [load_history(path) for path in paths]
    reference_x = runs[0][0]
    values = np.vstack([y for _, y in runs])
    mean = values.mean(axis=0)
    stderr = values.std(axis=0, ddof=1) / math.sqrt(values.shape[0])
    return reference_x, mean, stderr


def compact_y_limits(curves):
    values = np.concatenate([np.asarray(curve, dtype=np.float64) for curve in curves])
    lo, hi = float(values.min()), float(values.max())
    span = hi - lo
    if span <= 0.0:
        span = max(abs(lo), 1.0)
    pad = max(1e-6, LINE_Y_PADDING_FRACTION * span)
    return lo - pad, hi + pad


def uses_step_axis():
    return "step" in X_KEY.lower()


def apply_step_axis_format(axis):
    if uses_step_axis():
        axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _pos: f"{value / 1e5:g}"))
        # SET fontsize
        axis.tick_params(axis="x", labelsize=AXES_FONTSIZE)
        axis.tick_params(axis="y", labelsize=AXES_FONTSIZE)


In [239]:
def build_preset(name):
    selected = SELECTED_CONFIGS[name]
    results_root = selected["results_root"].rstrip("/")
    run_prefix = selected["run_prefix"]
    configs = {
        "ippo": selected["ippo"],
        "mappo": selected["mappo"],
        "pimac_v0": selected["pimac_v0"],
        "pimac_v6": selected["pc3d"],
    }
    series = []
    for algorithm in SERIES_ORDER:
        series.append({
            "label": ALGORITHM_LABELS[algorithm],
            "color": ALGORITHM_COLORS[algorithm],
            "glob": f"{results_root}/{algorithm}/{run_prefix}_{algorithm}_{configs[algorithm]}_s*/train_history.csv",
        })
    return {
        "title": selected["title"],
        "output_filename": selected["output_filename"],
        "stage_source": f"{results_root}/pimac_v6/{run_prefix}_pimac_v6_{selected['pc3d']}_s42/train_history.csv",
        "series": series,
    }


def plot_single_learning_curve(preset_name):
    MIDDLE_FIGURE = preset_name == "lbf_final_selected"
    LEFT_FIGURE = preset_name == "spread_final_selected"
    preset = build_preset(preset_name)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / preset["output_filename"]

    if MIDDLE_FIGURE:
        fig, axis = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT * 1.1), dpi=DPI)
    else:
        fig, axis = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), dpi=DPI)
    segments = load_stage_segments(REPO_ROOT / preset["stage_source"])
    if MERGE_IDENTICAL_ADJACENT_STAGES:
        segments = merge_adjacent_stage_segments(segments)

    for segment_index, segment in enumerate(segments):
        color = STAGE_FACE_COLORS[segment["indices"][0] % len(STAGE_FACE_COLORS)]
        axis.axvspan(segment["start"], segment["end"], color=color, alpha=STAGE_ALPHA, linewidth=0)
        if segment_index > 0:
            axis.axvline(segment["start"], color="#555555", linewidth=1.0, alpha=SEPARATOR_ALPHA, linestyle="--")
        axis.text(
            (segment["start"] + segment["end"]) / 2.0,
            0.98,
            segment["label"],
            transform=axis.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=STAGE_FONTSIZE,
            color="#444444",
            alpha=0.88,
        )

    z = NormalDist().inv_cdf(0.5 + CI_LEVEL / 2.0)
    mean_curves = []
    for item in preset["series"]:
        paths = sorted(REPO_ROOT.glob(item["glob"]))
        x, mean, stderr = aggregate_runs(paths)
        mean_curves.append(mean)
        axis.plot(x, mean, label=item["label"], color=item["color"], linewidth=2.0)
        axis.fill_between(x, mean - z * stderr, mean + z * stderr, color=item["color"], alpha=0.18)

    if COMPACT_Y_LIMITS:
        axis.set_ylim(*compact_y_limits(mean_curves))
    axis.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    if LEFT_FIGURE:
        axis.set_ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    apply_step_axis_format(axis)
    axis.grid(True, alpha=0.18)
    if PRINT_LEGEND and MIDDLE_FIGURE:
        axis.legend(ncol=len(preset["series"]), bbox_to_anchor=(0.5, 1.1), loc="center", fontsize=LEGEND_FONTSIZE, frameon=True)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight", dpi=DPI)
    plt.close(fig)

    paths = [output_path]
    return paths


In [240]:
saved_paths = []
for preset_name in PLOTS_TO_MAKE:
    saved_paths.extend(plot_single_learning_curve(preset_name))

for path in saved_paths:
    print(path)


/Users/akman/pimac/plotting/single_curves/spread_hard_final_learning_curves.png
/Users/akman/pimac/plotting/single_curves/lbf_hard_final_learning_curves.png
/Users/akman/pimac/plotting/single_curves/rware_final_learning_curves.png
